<a href="https://colab.research.google.com/github/johnchaparros83-stack/Unicafam_Modelos_Analiticos_Indra/blob/main/Sesion3_Integracion_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Integración de datos**

In [ ]:
# Importar librerias
import pandas as pd
import numpy as np
from io import StringIO

In [ ]:
#Crear archivo csv_clientes con separador ;
csv_clientes = """id_cliente;nombre;ciudad;segmento
C001;Ana Ruiz;Bogotá;Oro
C002;Luis Peña;Medellín;Plata
C003;Marta Gil;Cali;Oro
C004;Juan Ríos;Bogotá;Bronce
C005;Sofía Mora;Barranquilla;Plata"""

In [ ]:
#Leer archivo, separador ;
clientes = pd.read_csv(StringIO(csv_clientes), sep=";")
#Imprimir numero de columnas y filas
print(clientes.shape)
#Imprimir los primeros 3 registros
print(clientes.head(3))

(5, 4)
  id_cliente     nombre    ciudad segmento
0       C001   Ana Ruiz    Bogotá      Oro
1       C002  Luis Peña  Medellín    Plata
2       C003  Marta Gil      Cali      Oro


In [ ]:
#Integrar un archivo JSON vía API
respuesta_api = [
    {"pedido": 101, "cliente": "C001", "items": 3,
     "pago": {"medio": "Tarjeta", "valor": 150000}},
    {"pedido": 102, "cliente": "C003", "items": 1,
     "pago": {"medio": "PSE", "valor": 80000}},
    {"pedido": 103, "cliente": "C001", "items": 2,
     "pago": {"medio": "Efectivo", "valor": 45000}},
    {"pedido": 104, "cliente": "C009", "items": 5,
     "pago": {"medio": "Tarjeta", "valor": 210000}},
]
pedidos = pd.json_normalize(respuesta_api)
print(pedidos)


   pedido cliente  items pago.medio  pago.valor
0     101    C001      3    Tarjeta      150000
1     102    C003      1        PSE       80000
2     103    C001      2   Efectivo       45000
3     104    C009      5    Tarjeta      210000


# **MERGE DATOS**

In [ ]:
inner = pd.merge(clientes, pedidos, left_on="id_cliente",
                 right_on="cliente", how="inner")
left = pd.merge(clientes, pedidos, left_on="id_cliente",
                right_on="cliente", how="left")
print("INNER:", inner.shape[0], "filas")
print("LEFT :", left.shape[0], "filas")
#Contar nulos y sacar la lista
sin_compra = left[left["pedido"].isna()]["nombre"].tolist()
print("Clientes sin compras:", sin_compra)

  id_cliente      nombre        ciudad segmento
0       C001    Ana Ruiz        Bogotá      Oro
1       C002   Luis Peña      Medellín    Plata
2       C003   Marta Gil          Cali      Oro
3       C004   Juan Ríos        Bogotá   Bronce
4       C005  Sofía Mora  Barranquilla    Plata
   pedido cliente  items pago.medio  pago.valor
0     101    C001      3    Tarjeta      150000
1     102    C003      1        PSE       80000
2     103    C001      2   Efectivo       45000
3     104    C009      5    Tarjeta      210000
INNER: 3 filas
LEFT : 6 filas
Clientes sin compras: ['Luis Peña', 'Juan Ríos', 'Sofía Mora']


In [ ]:
inner

,id_cliente,nombre,ciudad,segmento,pedido,cliente,items,pago.medio,pago.valor
0,C001,Ana Ruiz,Bogotá,Oro,101,C001,3,Tarjeta,150000
1,C001,Ana Ruiz,Bogotá,Oro,103,C001,2,Efectivo,45000
2,C003,Marta Gil,Cali,Oro,102,C003,1,PSE,80000


In [ ]:
left

,id_cliente,nombre,ciudad,segmento,pedido,cliente,items,pago.medio,pago.valor
0,C001,Ana Ruiz,Bogotá,Oro,101.0,C001,3.0,Tarjeta,150000.0
1,C001,Ana Ruiz,Bogotá,Oro,103.0,C001,2.0,Efectivo,45000.0
2,C002,Luis Peña,Medellín,Plata,NaN,NaN,NaN,NaN,NaN
3,C003,Marta Gil,Cali,Oro,102.0,C003,1.0,PSE,80000.0
4,C004,Juan Ríos,Bogotá,Bronce,NaN,NaN,NaN,NaN,NaN
5,C005,Sofía Mora,Barranquilla,Plata,NaN,NaN,NaN,NaN,NaN


# **Consolidación de datos**

In [ ]:
np.random.seed(1)
def ventas_mes(mes):
    return pd.DataFrame({
        "mes": mes,
        "sucursal": ["Norte", "Sur", "Centro"],
        "ventas": np.random.randint(80, 200, 3) * 1e6})

consolidado = pd.concat([ventas_mes("Ene"), ventas_mes("Feb"),
                         ventas_mes("Mar")], ignore_index=True)
print(consolidado.shape)
print(consolidado.tail(3))


(9, 3)
   mes sucursal       ventas
6  Mar    Norte   85000000.0
7  Mar      Sur  159000000.0
8  Mar   Centro  144000000.0


In [ ]:
resumen = consolidado.groupby("sucursal").agg(
    total=("ventas", "sum"),
    promedio=("ventas", "mean"),
    meses=("mes", "count"))
resumen = (resumen / [1e6, 1e6, 1]).round(1)
print(resumen.sort_values("total", ascending=False))


          total  promedio  meses
sucursal                        
Sur       435.0     145.0    3.0
Centro    391.0     130.3    3.0
Norte     354.0     118.0    3.0


In [ ]:
ancho = consolidado.pivot_table(index="sucursal", columns="mes",
                                values="ventas") / 1e6
ancho = ancho[["Ene", "Feb", "Mar"]]
print(ancho)

largo = ancho.reset_index().melt(id_vars="sucursal",
                                 var_name="mes", value_name="ventas_M")
print(largo.head(4))


mes         Ene    Feb    Mar
sucursal                     
Centro     92.0  155.0  144.0
Norte     117.0  152.0   85.0
Sur       187.0   89.0  159.0
  sucursal  mes  ventas_M
0   Centro  Ene      92.0
1    Norte  Ene     117.0
2      Sur  Ene     187.0
3   Centro  Feb     155.0


# **Ejercicio 1**

In [28]:
#Crear 3 clientes nuevos
nuevos_clientes = pd.DataFrame({
    "id_cliente": ["C006", "C007", "C008"],
    "nombre": ["Carlos", "Laura", "Andrés"]
})

clientes = pd.concat(
    [clientes, nuevos_clientes],
    ignore_index=True
)
clientes

,id_cliente,nombre,ciudad,segmento
0,C001,Ana Ruiz,Bogotá,Oro
1,C002,Luis Peña,Medellín,Plata
2,C003,Marta Gil,Cali,Oro
3,C004,Juan Ríos,Bogotá,Bronce
4,C005,Sofía Mora,Barranquilla,Plata
5,C006,Carlos,NaN,NaN
6,C007,Laura,NaN,NaN
7,C008,Andrés,NaN,NaN
8,C006,Carlos,NaN,NaN
9,C007,Laura,NaN,NaN


In [41]:
 # LEFT JOIN con pedidos y lista los clientes sin compras.

resultado_left = pd.merge(
    clientes,
    pedidos,
    left_on="id_cliente",
    right_on="cliente",
    how="left",
    indicator=True
)

print(sin_compras)

   id_cliente      nombre        ciudad segmento  pedido cliente  items  \
2        C002   Luis Peña      Medellín    Plata     NaN     NaN    NaN   
4        C004   Juan Ríos        Bogotá   Bronce     NaN     NaN    NaN   
5        C005  Sofía Mora  Barranquilla    Plata     NaN     NaN    NaN   
6        C006      Carlos           NaN      NaN     NaN     NaN    NaN   
7        C007       Laura           NaN      NaN     NaN     NaN    NaN   
8        C008      Andrés           NaN      NaN     NaN     NaN    NaN   
9        C006      Carlos           NaN      NaN     NaN     NaN    NaN   
10       C007       Laura           NaN      NaN     NaN     NaN    NaN   
11       C008      Andrés           NaN      NaN     NaN     NaN    NaN   
12       C006      Carlos           NaN      NaN     NaN     NaN    NaN   
13       C007       Laura           NaN      NaN     NaN     NaN    NaN   
14       C008      Andrés           NaN      NaN     NaN     NaN    NaN   

   pago.medio  pago.valo

In [43]:
 # LEFT JOIN con pedidos y lista los clientes sin compras.

resultado_left = pd.merge(
    clientes,
    pedidos,
    left_on="id_cliente",
    right_on="cliente",
    how="left",
    indicator=True
)

sin_compras = resultado_left[
    resultado_left["_merge"] == "left_only"
]

print(sin_compras)

   id_cliente      nombre        ciudad segmento  pedido cliente  items  \
2        C002   Luis Peña      Medellín    Plata     NaN     NaN    NaN   
4        C004   Juan Ríos        Bogotá   Bronce     NaN     NaN    NaN   
5        C005  Sofía Mora  Barranquilla    Plata     NaN     NaN    NaN   
6        C006      Carlos           NaN      NaN     NaN     NaN    NaN   
7        C007       Laura           NaN      NaN     NaN     NaN    NaN   
8        C008      Andrés           NaN      NaN     NaN     NaN    NaN   
9        C006      Carlos           NaN      NaN     NaN     NaN    NaN   
10       C007       Laura           NaN      NaN     NaN     NaN    NaN   
11       C008      Andrés           NaN      NaN     NaN     NaN    NaN   
12       C006      Carlos           NaN      NaN     NaN     NaN    NaN   
13       C007       Laura           NaN      NaN     NaN     NaN    NaN   
14       C008      Andrés           NaN      NaN     NaN     NaN    NaN   

   pago.medio  pago.valo

In [45]:
# OUTER JOIN con indicator true

resultado_outer = pd.merge(
    clientes,
    pedidos,
    left_on="id_cliente",
    right_on="cliente",
    how="outer",
    indicator=True
)

conteo = resultado_outer["_merge"].value_counts()

print(resultado_outer)
print(conteo)

   id_cliente      nombre        ciudad segmento  pedido cliente  items  \
0        C001    Ana Ruiz        Bogotá      Oro   101.0    C001    3.0   
1        C001    Ana Ruiz        Bogotá      Oro   103.0    C001    2.0   
2        C002   Luis Peña      Medellín    Plata     NaN     NaN    NaN   
3        C003   Marta Gil          Cali      Oro   102.0    C003    1.0   
4        C004   Juan Ríos        Bogotá   Bronce     NaN     NaN    NaN   
5        C005  Sofía Mora  Barranquilla    Plata     NaN     NaN    NaN   
6        C006      Carlos           NaN      NaN     NaN     NaN    NaN   
7        C006      Carlos           NaN      NaN     NaN     NaN    NaN   
8        C006      Carlos           NaN      NaN     NaN     NaN    NaN   
9        C007       Laura           NaN      NaN     NaN     NaN    NaN   
10       C007       Laura           NaN      NaN     NaN     NaN    NaN   
11       C007       Laura           NaN      NaN     NaN     NaN    NaN   
12       C008      Andrés

In [46]:
# Lista final Marketing/los que no compraron

lista_marketing = sin_compras[
    ["id_cliente"]
]

print("CLIENTES PARA CAMPAÑA DE MARKETING")
print(lista_marketing)

CLIENTES PARA CAMPAÑA DE MARKETING
   id_cliente
2        C002
4        C004
5        C005
6        C006
7        C007
8        C008
9        C006
10       C007
11       C008
12       C006
13       C007
14       C008


# **Ejercicio 2**

In [47]:
#pasar  formato pivote

tabla = consolidado.pivot_table(
    index="sucursal",
    columns="mes",
    values="ventas",
    aggfunc="sum"
)

print(tabla)

mes               Ene          Feb          Mar
sucursal                                       
Centro     92000000.0  155000000.0  144000000.0
Norte     117000000.0  152000000.0   85000000.0
Sur       187000000.0   89000000.0  159000000.0


In [49]:
#Calcular crecimiento %

tabla = tabla[["Ene", "Feb", "Mar"]]

tabla["crecimiento_%"] = (
    (tabla["Mar"] - tabla["Ene"]) / tabla["Ene"]
) * 100

print(tabla)

mes               Ene          Feb          Mar  crecimiento_%
sucursal                                                      
Centro     92000000.0  155000000.0  144000000.0      56.521739
Norte     117000000.0  152000000.0   85000000.0     -27.350427
Sur       187000000.0   89000000.0  159000000.0     -14.973262


In [50]:
# Ordenar de mayor a menor
tabla = tabla.sort_values(
    by="crecimiento_%",
    ascending=False
)

print(tabla)


mes               Ene          Feb          Mar  crecimiento_%
sucursal                                                      
Centro     92000000.0  155000000.0  144000000.0      56.521739
Sur       187000000.0   89000000.0  159000000.0     -14.973262
Norte     117000000.0  152000000.0   85000000.0     -27.350427


In [51]:
#Identificar sucursal lider
lider = tabla["crecimiento_%"].idxmax()

print("La sucursal líder es:", lider)
print("Crecimiento:", round(tabla["crecimiento_%"].max(), 2), "%")


La sucursal líder es: Centro
Crecimiento: 56.52 %


# **Ejercicio 3**